# Encoder–Decoder 구조

- Encoder–Decoder 구조는 어떤 형태의 입력 시퀀스를 받아 **의미를 해석**한 뒤, 새로운 **출력 시퀀스를 생성**해야 하는 거의 모든 AI 문제를 해결하는 딥러닝 모델 구조다.
- 이 구조는 Encoder와 Decoder 두개의 딥러닝 모델을 연결한 구조로 **입력 데이터를 하나의 표현으로 압축한 뒤, 이를 다시 출력 데이터로 변환하는 방식**으로 동작한다.

- **Encoder Network**
  - 입력 데이터를 해석(이해)하는 역할을 수행한다.
  - 입력 시퀀스에 담긴 의미적 정보를 하나의 고정된 벡터 형태로 요약한다.

- **Decoder Network**
  - Encoder가 생성한 요약 정보를 바탕으로 최종 출력을 생성한다.
  - 즉, Encoder의 “이해 결과”를 이용해 새로운 시퀀스를 만들어낸다.

## Seq2Seq (Sequence-to-Sequence)

Seq2Seq 모델은 **Encoder–Decoder 구조를 RNN(Recurrent Neural Network) 계열에 적용한 대표적인 시퀀스 변환 모델**이다.  
입력과 출력이 모두 “시퀀스(sequence)” 형태라는 점에서 *Sequence-to-Sequence*라는 이름이 붙었다.

### Encoder의 역할: 입력 시퀀스 이해 및 Context Vector 생성

Encoder는 입력으로 들어온 **전체 시퀀스**(sequence)를 순차적으로 처리한 뒤,  그 의미를 **하나의 고정 길이 벡터**(Vector)로 압축하여 출력한다.  
이 벡터를 **Context Vector**(컨텍스트 벡터)라고 한다.
- **Context Vector란?**  
  - 입력 시퀀스 전체의 의미, 문맥, 핵심 정보를 요약해 담고 있는 벡터 표현이다.
  - **기계 번역**(Machine Translation)의 경우  
    - 번역할 원문 문장에서 **번역 결과를 생성하는 데 필요한 핵심 의미 정보**(feature)
  - **챗봇**(Chatbot)의 경우  
    - 사용자가 입력한 질문에서 **적절한 답변을 생성하는 데 필요한 의미 정보**(feature)

### Decoder의 역할: Context Vector를 바탕으로 출력 시퀀스 생성

Decoder는 Encoder가 출력한 **Context Vector를 입력으로 받아**, 이를 바탕으로 **목표 출력 시퀀스**를 한 토큰(token)씩 순차적으로 생성한다.

- **기계 번역**(Machine Translation)의 경우  
  - 입력 문장의 의미를 반영한 **번역 문장**을 생성한다.
- **챗봇**(Chatbot)  
  - 질문에 대한 **자연스러운 답변 문장**을 생성한다.

Decoder는 매 시점(time step)마다
  - 이전에 생성한 단어
  - 그리고 Context Vector에 담긴 입력 문맥
을 함께 고려하여 다음 단어를 예측한다.


![seq2seq](figures/seq2seq.png)

# Seq2Seq 를 이용한 Chatbot 모델 구현
- Encoder를 이용해 질문의 특성을 추출하고 Decoder를 이용해 답변을 생성한다.

# Chatbot Dataset

- https://github.com/songys/Chatbot_data
- columns
    - Q: 질문
    - A: 답
    - label: 일상다반사 0, 이별(부정) 1, 사랑(긍정) 2
- **Download**

![dataset](figures/chatbot.png)

# Chatbot Dataset Loading 및 확인

## 데이터셋 다운로드 및 확인

In [2]:
url = "https://raw.githubusercontent.com/songys/Chatbot_data/refs/heads/master/ChatbotData.csv"

In [3]:
import requests
import os

os.makedirs("data/chatbot", exist_ok=True)
res = requests.get(url)
if res.status_code == 200:
    with open("data/chatbot/chatbot_data.csv", "wt", encoding="utf-8") as fw:
        fw.write(res.text)
else:
    print("다운로드 실패 - ", res.status_code)

In [4]:
import pandas as pd

df = pd.read_csv("data/chatbot/chatbot_data.csv", encoding="utf-8")
df.shape

(11823, 3)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11823 entries, 0 to 11822
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Q       11823 non-null  str  
 1   A       11823 non-null  str  
 2   label   11823 non-null  int64
dtypes: int64(1), str(2)
memory usage: 277.2 KB


In [6]:
# label 대화분류 컬럼 -> 제거
df.drop(columns=["label"], inplace=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11823 entries, 0 to 11822
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Q       11823 non-null  str  
 1   A       11823 non-null  str  
dtypes: str(2)
memory usage: 184.9 KB


In [8]:
df.head(10)

,Q,A
0,12시 땡!,하루가 또 가네요.
1,1지망 학교 떨어졌어,위로해 드립니다.
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.
4,PPL 심하네,눈살이 찌푸려지죠.
5,SD카드 망가졌어,다시 새로 사는 게 마음 편해요.
6,SD카드 안돼,다시 새로 사는 게 마음 편해요.
7,SNS 맞팔 왜 안하지ㅠㅠ,잘 모르고 있을 수도 있어요.
8,SNS 시간낭비인 거 아는데 매일 하는 중,시간을 정하고 해보세요.
9,SNS 시간낭비인데 자꾸 보게됨,시간을 정하고 해보세요.


In [7]:
df.tail(10)

,Q,A
11813,회사에 좋아하는 남자가 생겼어 어떡하지?,사랑하기 힘든 관계인가봐요.
11814,회사에서 어떤 사람이랑 자꾸 눈 마추쳐.,눈 마주치는 게 우연인지 잘 살펴 보세요.
11815,회식 중이라고 하는데 연락이 안돼.,정신 없이 바쁠지도 몰라요. 조금만 더 기다려보고 물어보는게 좋을 것 같아요.
11816,회식하는데 나만 챙겨줘. 썸임?,호감이 있을 수도 있어요. 그렇지만 조금 더 상황을 지켜보세요.
11817,후회 없이 사랑하고 싶어,진심으로 다가가 보세요.
11818,훔쳐보는 것도 눈치 보임.,티가 나니까 눈치가 보이는 거죠!
11819,훔쳐보는 것도 눈치 보임.,훔쳐보는 거 티나나봐요.
11820,흑기사 해주는 짝남.,설렜겠어요.
11821,힘든 연애 좋은 연애라는게 무슨 차이일까?,잘 헤어질 수 있는 사이 여부인 거 같아요.
11822,힘들어서 결혼할까봐,도피성 결혼은 하지 않길 바라요.


In [8]:
# 결측치
df.isnull().sum()

Q    0
A    0
dtype: int64

# Dataset, DataLoader 정의

## Tokenization

### Subword방식

In [9]:
question_texts = df['Q']
answer_texts = df['A']
# 토큰화를 위해서 "질문 + 답변"을 합쳐서 하나의 문서로 만든다.
all_texts = list(question_texts + " " + answer_texts) # 시리즈 + 시리즈 : 원소별 연산.

In [10]:
all_texts[:5]

['12시 땡! 하루가 또 가네요.',
 '1지망 학교 떨어졌어 위로해 드립니다.',
 '3박4일 놀러가고 싶다 여행은 언제나 좋죠.',
 '3박4일 정도 놀러가고 싶다 여행은 언제나 좋죠.',
 'PPL 심하네 눈살이 찌푸려지죠.']

In [11]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=30_000,
    min_frequency=2,
    continuing_subword_prefix="##", # 단어 중간에 붙는 subword일 경우 붙이는 접두어. 
    special_tokens=["<unk>", "<pad>", "<sos>"] # <sos> : 문장의 시작을 알리는 토큰. 
)

# 토큰나이저 학습
tokenizer.train_from_iterator(all_texts, trainer=trainer)

In [12]:
print(tokenizer.get_vocab_size())

15578


In [18]:
e = tokenizer.encode("3박4일 정도 놀러가고 싶다")
e
# e.tokens
e.ids

[15122, 2892, 5236, 2331]

### Tokenizer 저장

In [15]:
# import os
os.makedirs("saved_model/chatbot_seq2seq", exist_ok=True)
tokenizer.save("saved_model/chatbot_seq2seq/chatbot_bpe.json")

## Dataset, DataLoader 정의


### Dataset 정의 및 생성
- 모든 문장의 토큰 수는 동일하게 맞춰준다.
    - DataLoader는 batch 를 구성할 때 batch에 포함되는 데이터들의 shape이 같아야 한다. 그래야 하나의 batch로 묶을 수 있다.
    - 문장의 최대 길이를 정해주고 **최대 길이보다 짧은 문장은 `<PAD>` 토큰을 추가**하고 **최대길이보다 긴 문장은 최대 길이에 맞춰 짤라준다.**

In [16]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [29]:
class A:

    ## 메소드, instance 변수에  __시작: 클래스 내에서만 사용.
    def __test(self):
        print("A.test()")

    def b(self):
        self.__test()

a = A()
a._A__test()
dir(a)

A.test()


['_A__test',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'b']

In [30]:
# Dataset 정의
class ChatbotDataset(Dataset):

    def __init__(self, question_texts, answer_texts, max_length, tokenizer):
        """
        Args:
            question_texts: list[str] - 질문 text들. ["질문1", "질문2", ...]
            answer_texts: list[str] - 답변 text들. ["답변1", "답변2", ...]
            max_length: int - 개별 질문, 답변의 토큰 수
            tokenizer: Tokenizer
        """
        self.max_length = max_length
        self.tokenizer = tokenizer
        # [Tensor([토큰 id, ...])]
        # "3박 4일 휴가동안 미국에 여행갈 예정이다." -> [100, 200, 222, 321, ...] : 토큰개수: Max_length
        self.question_texts = [self.__process_sequence(q) for q in question_texts]
        self.answer_texts = [self.__process_sequence(a) for a in answer_texts]
        # list[LongTensor]  # 자료구조 타입힌트 - a:list[str] b:tuple[int]

    def __process_sequence(self, text:str):
        """한 문장을 받아서 max_length 크기의 토큰 tensor로 반환."""
        # 토큰화
        encoding = self.tokenizer.encode(text)
        # max_length 크기에 맞추기 (모자라면 <pad> 추가, 넘치면 잘라내기.)
        tokens = self.__pad_token_sequence(encoding.ids)
        return torch.tensor(tokens, dtype=torch.int64)

    def __pad_token_sequence(self, token_sequence):
        """token sequence: [[토큰 ID, ...], [.....]]을 max_length에 크기를 맞춰준다."""
        pad_token = self.tokenizer.token_to_id("<pad>") # Padding 토큰의 id 조회.
        seq_length = len(token_sequence)
        if seq_length >= self.max_length: # max_length크기로 줄이기
            result = token_sequence[:self.max_length]
        else:
            result = token_sequence + [pad_token] * (self.max_length - seq_length)
        
        return result

    def __len__(self):
        return len(self.question_texts)

    def __getitem__(self, index):
        """ index의 X(question), y(answer)을 튜플로 반환"""

        q = self.question_texts[index]
        a = self.answer_texts[index]
        return q, a

In [31]:
# max_length - QA중에서 가장 긴 토큰으로 맞춘다.
max([len(tokenizer.encode(sent).ids) for sent in question_texts])

18

In [32]:
max([len(tokenizer.encode(sent).ids) for sent in answer_texts])

26

In [33]:
max_length = 26
dataset = ChatbotDataset(
    question_texts,
    answer_texts,
    max_length,
    tokenizer
)

In [34]:
question_texts[1200], answer_texts[1200]

('대리운전 불렀어', '잘했어요.')

In [35]:
dataset[1200]

(tensor([5319, 1272, 1617,  591, 7700,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1]),
 tensor([4538,    8,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1]))

In [47]:
len(dataset)

11823

### Trainset / Testset 나누기
train : test = 0.95 : 0.05

In [36]:
train_size = int(len(dataset) * 0.95)
test_size = len(dataset) - train_size

train_size, test_size

(11231, 592)

In [37]:
trainset, testset = random_split(dataset, [train_size, test_size])
len(trainset), len(testset)

(11231, 592)

### DataLoader 생성

In [38]:
train_loader = DataLoader(trainset, batch_size=100, shuffle=True, drop_last=True)
test_loader = DataLoader(testset, batch_size=100)

In [39]:
len(train_loader), len(test_loader)

(112, 6)

In [55]:
df.head()

,Q,A
0,12시 땡!,하루가 또 가네요.
1,1지망 학교 떨어졌어,위로해 드립니다.
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.
4,PPL 심하네,눈살이 찌푸려지죠.


# 모델 정의

## Encoder
Encoder는 하나의 Vector를 생성하며 그 Vector는 **입력 문장의 의미**를 N 차원 공간 저장하고 있다. 이 Vector를 **Context Vector** 라고 한다.    
![encoder](figures/seq2seq_encoder.png)

In [41]:
pad_id = tokenizer.token_to_id("<pad>")
class Encoder(nn.Module):
    def __init__(
            self,
            vocab_size: int, # 어휘수. tokenizer의 vocab_size 받는다.
            embedding_dim: int, # Embedding Vector의 차원수.
            hidden_size: int,   # GRU의 hidden 개수 (추출할 특성개수) 
            bidirectional: bool=True, # GRU 양방향여부. Encoder는 문서를 입력받기 때문에 양방향설정을 한다.
            num_layers: int=1, # GRU의 layer stack 수.
            dropout: float=0.2 # Dropout 비율
    ):
        super().__init__()
        ## EmbeddingLayer -> GRU (-> Decoder)
        # Embedding Layer
        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=1
        )
        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
            # batch_first=False
        )
        self.vocab_size = vocab_size
        
    def forward(self, X):
        # X.shape: [batch_size:문서개수, seq_length:토큰개수] -> embedding 
        ##       -> embedding_vector.shape: [batch_size, seq_length, embedding_dim]
        embedding_vector = self.embedding(X)
        
        # batch_size 축 <-> seq_length 축 
        ##### (batch_size, seq_length, embedding_dim)->(seq_length, batch_size, embedding_dim)
        embedding_vector = embedding_vector.transpose(1, 0)

        out, hidden = self.gru(embedding_vector)
        #out: 모든 time step의 hidden state들
        #hidden: 마지막 time step의 hidden state
        return out, hidden

In [43]:
from torchinfo import summary
v_size = 1000 # 단어수
e_size = 100  # embedding dim
h_size = 20   # hidden size


dummy_input = torch.randint(1000, size=(256, 26), dtype=torch.int64)  # (batch, 토큰수)
summary(Encoder(v_size, e_size, h_size), input_data=dummy_input)

Layer (type:depth-idx)                   Output Shape              Param #
Encoder                                  [26, 256, 40]             --
├─Embedding: 1-1                         [256, 26, 100]            100,000
├─GRU: 1-2                               [26, 256, 40]             14,640
Total params: 114,640
Trainable params: 114,640
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 123.04
Input size (MB): 0.05
Forward/backward pass size (MB): 7.45
Params size (MB): 0.46
Estimated Total Size (MB): 7.97

## Decoder
- Encoder의 출력(context vector)를 받아서 답변 결과 sequence를 출력한다.
- Decoder는 매 time step의 입력으로 **이전 time step에서 예상한 단어와 hidden state값이** 입력된다.
- Decoder의 처리결과 hidden state를 Estimator(Linear+Softmax)로 입력하여 **입력 단어에 대한 번역 단어가 출력된다.** (이 출력단어가 다음 step의 입력이 된다.)
    - Decoder의 첫 time step 입력은 문장의 시작을 의미하는 <SOS>(start of string) 토큰이고 hidden state는 context vector(encoder 마지막 hidden state) 이다.

![decoder](figures/seq2seq_decoder.png)

In [44]:
class Decoder(nn.Module):

    def __init__(
            self,
            vocab_size:int,   # 어휘사전 크기
            embedding_dim:int,# embedding vector 차원수 
            hidden_size:int, # GRU의 hidden state의 크기(hidden_size)
            num_layers:int=1, # GRU의 layer stack 수
            dropout:float=0.2 # GRU의 dropout 비율
    ):
        # Decoder는 자기 회귀(auto regressive)모델 -> bidirectional=False 고정.
        super().__init__()
        # 임베딩 모델
        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=1
        )
        # GRU
        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            # bidirection=False # default: False
            # batch_first=False # default: False 입력 shape:(seq_len, batch, feature)
        )
        # Linear - 다음 단어(토큰)을 추론하는 분류기.
        self.classifier = nn.Linear(
            hidden_size, # 입력: GRU의 출력 hidden state개수
            vocab_size   # 출력: 다중분류 - 어휘사전의 단어들의 다음 단어일 확률.
        )

    def forward(self, X, hidden):
        """
        Args:
            X: 한개 토큰. shape: [batch]. ex) [1000(첫번째 문서의 답변토큰), 2000(두번째 문서), 1231, ..]
            hidden: 이전 timestep까지의 처리 hidden state. 
                    첫번째 timestep일 경우 Encoder의 context vector(encoder의 마지막 timestep의 hidden state)
                    [seq_length: 1, batch_size, hidden_size]
        """
        # 1. X shape: [batch] -> [batch, 1]
        X = X.unsqueeze(1)
        # embedding 처리: [batch, 1: seq_length] -> [batch, 1, embedding_dim]
        embedding_vector = self.embedding(X)
        
        # 1축 <-> 0축
        embedding_vector = embedding_vector.transpose(1, 0) # [1, batch, embeding_dim]

        # gru
        out, hidden = self.gru(embedding_vector, hidden)
        # out(모든 타임스텝) shape: [seq_len(1), batch_size, hidden_size]
        # hidden(마지막 타임스텝) shape: [num_layer * bidirectional(1), batch_size, hidden_size]

        # linear(classifier) => 다음단어 예측기(분류기)
        last_out = self.classifier(out[-1])

        return last_out, hidden

In [45]:
b_size = 256
v_size = 1000
h_size = 20
e_size = 100
dummy_input = torch.randint(v_size, [b_size], dtype=torch.int64)  # 한개 단어
dummy_hidden = torch.randn(size=(1, b_size, h_size), dtype=torch.float32)
dummy_decoder = Decoder(v_size, e_size, h_size)

summary(dummy_decoder, input_data=(dummy_input, dummy_hidden))

Layer (type:depth-idx)                   Output Shape              Param #
Decoder                                  [256, 1000]               --
├─Embedding: 1-1                         [256, 1, 100]             100,000
├─GRU: 1-2                               [1, 256, 20]              7,320
├─Linear: 1-3                            [256, 1000]               21,000
Total params: 128,320
Trainable params: 128,320
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 32.85
Input size (MB): 0.02
Forward/backward pass size (MB): 2.29
Params size (MB): 0.51
Estimated Total Size (MB): 2.83

## Seq2Seq 모델

- Encoder - Decoder 를 Layer로 가지며 Encoder로 질문의 feature를 추출하고 Decoder로 답변을 생성한다.

### Teacher Forcing
- **Teacher forcing** 기법은, RNN계열 모델이 다음 단어를 예측할 때, 이전 timestep에서 예측된 단어를 입력으로 사용하는 대신 **실제 정답 단어(ground truth) 단어를** 입력으로 사용하는 방법이다.
    - 모델은 이전 시점의 출력 단어를 다음 시점의 입력으로 사용한다. 그러나 모델이 학습할 때 초반에는 정답과 많이 다른 단어가 생성되어 엉뚱한 입력이 들어가 학습이 빠르게 되지 않는 문제가 있다.
- **장점**
    - **수렴 속도 증가**: 정답 단어를 사용하기 때문에 모델이 더 빨리 학습할 수있다.
    - **안정적인 학습**: 초기 학습 단계에서 모델의 예측이 불안정할 때, 잘못된 예측으로 인한 오류가 다음 단계로 전파되는 것을 막아줍니다.
- **단점**
    - **노출 편향(Exposure Bias) 문제:** 실제 예측 시에는 정답을 제공할 수 없으므로 모델은 전단계의 출력값을 기반으로 예측해 나가야 한다. 학습 과정과 추론과정의 이러한 차이 때문에 모델의 성능이 떨어질 수있다.
        - 이런 문제를 해결하기 학습 할 때 **Teacher forcing을 random하게 적용하여 학습시킨다.**

![seq2seq](figures/seq2seq.png)

In [91]:
import random
sos_token = tokenizer.token_to_id("<sos>")
# sos_token

class ChatbotSeq2Seq(nn.Module):

    def __init__(self, encoder, decoder, device="cpu"):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder 
        self.device = device

    def forward(self, inputs, outputs, teacher_forcing_rate=0.99):
        """
        Args:
            inputs (tensor): 질문. (batch, seq_length)
            outputs(tensor): 답변 정답. (batch, seq_length), teacher forcing 적용때 사용.
            teacher_forcing_rate(float): teacher forcing이 적용될 확률(랜덤하게 적용.)
        """
        # 1개 문서(데이터)가 입력될 때 1차원으로 들어 올 수있다. 
        # 1차원이면 2차원으로 변환. [seq_length] -> [1, seq_length]
        if inputs.dim() == 1:
            inputs = inputs.unsqueeze(0)
        
        if outputs.dim() == 1:
            outputs = outputs.unsqueeze(0)

        # output_length: 생성하는 답변의 토큰 개수를 정답의 토큰 개수에 맞춘다.
        batch_size, output_length = outputs.shape
        output_vocab_size = self.encoder.vocab_size  # 어휘수
        
        # 모델이 생성한 문서(답변)을 저장할 Tensor
        ## "나는", "소년", "이다"
        # [
        #    [모든 어휘들이 다음단어일 확률 - vocab_size], # "나는" 의 확률에 제일 높을 것.
        #    [모든 어휘들이 다음단어일 확률], - "소년"
        #    [모든 어휘들이 다음단어일 확률] - "이다"
        #]
        predicted_outputs = torch.zeros(
            output_length, # seq_length 
            batch_size,
            output_vocab_size # 모델이 추론한 모든 어휘가 그(다음) 단어일 확률
        ).to(self.device)

        #######################################################
        # 추론
        # 1. encoder를 통해서 context vector 추출. 한번 처리
        # 2. decoder를 통해서 답변 문장(문서) 생성. 하나씩 처리
        #######################################################
        
        # 1. context vector 추출. 
        ##### encoder_out: 모든 타임스텝의 hidden state (seq_length, batch, hidden_size*2)
        encoder_out, _ = self.encoder(inputs)
        
        # Decoder 첫번째 timestep의 입력할 값들을 생성
        ## 첫번째 timestep에 입력할 hidden state (context vector)
        decoder_hidden = encoder_out[-1].unsqueeze(0)  
        ## 첫번째 timestep의 입력값 (<sos> 토큰 ID)
        decoder_input = torch.full([batch_size], fill_value=sos_token, device=self.device)

        # 2. Decoder를 이용해서 한 단어(토큰)씩 생성. - decoder(현재 ts의 토큰, hidden-이전까지특성)
        for t in range(output_length):
            # decoder_out: 다음단어 - 어휘사전의 단어들이 다음 단어일 확률. [batch, voca_size]
            # decoder_hidden: hidden state
            decoder_out, decoder_hidden = self.decoder(decoder_input, decoder_hidden)

            # 다음 timestep의 input을 생성 -> 
            ## 모델이 예측한 단어 if teacher forcing을 안한다면 else 정답(outputs)단어
            teacher_forcing:bool = teacher_forcing_rate > random.random()
            teacher_forcing_rate *= 0.99  # teacher forcing rate 가 True일 확률을 줄여나간다.

            ## 모델이 예측한 다음 단어 토큰 ID를 생성
            top1 = decoder_out.argmax(dim=-1)
            decoder_input = outputs[:, t] if teacher_forcing else top1

            # decoder_out: 어휘사전의 단어들이 다음 단어일 확률.
            predicted_outputs[t] = decoder_out
        
        return predicted_outputs.transpose(1, 0) # seq_length <-> batch

# 학습

## 모델생성

In [92]:
# 파라미터 정의
vocab_size = tokenizer.get_vocab_size()
encoder_bidirectional = True
encoder_hidden_size = 200

# Decoder의 첫번째 hidden state == encoder의 마지막 hidden state(context vector)
decoder_hidden_size = encoder_hidden_size * 2 if encoder_bidirectional else encoder_hidden_size

embedding_dim = 256
teacher_forcing_rate = 0.9

In [93]:
encoder = Encoder(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_size=encoder_hidden_size,
    num_layers=1, 
    bidirectional=encoder_bidirectional
)
decoder = Decoder(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_size=decoder_hidden_size,
    num_layers=1
)
seq2seq = ChatbotSeq2Seq(encoder, decoder, device)

In [94]:
d_input = torch.randint(1000, (100, 26), dtype=torch.int64)
summary(seq2seq, input_data=(d_input, d_input))

Layer (type:depth-idx)                   Output Shape              Param #
ChatbotSeq2Seq                           [100, 26, 15578]          --
├─Encoder: 1-1                           [26, 100, 400]            --
│    └─Embedding: 2-1                    [100, 26, 256]            3,987,968
│    └─GRU: 2-2                          [26, 100, 400]            549,600
├─Decoder: 1-2                           [100, 15578]              --
│    └─Embedding: 2-3                    [100, 1, 256]             3,987,968
│    └─GRU: 2-4                          [1, 100, 400]             789,600
│    └─Linear: 2-5                       [100, 15578]              6,246,778
├─Decoder: 1-3                           [100, 15578]              (recursive)
│    └─Embedding: 2-6                    [100, 1, 256]             (recursive)
│    └─GRU: 2-7                          [1, 100, 400]             (recursive)
│    └─Linear: 2-8                       [100, 15578]              (recursive)
├─Decoder: 1-4    

## loss함수, optimizer

In [95]:
lr = 0.001
model = seq2seq.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

## train/evaluation 함수 정의

### train 함수정의

In [86]:
# 1 에폭학습 함수
def train(model, dataloader, optimizer, loss_fn, device, teacher_forcing_rate=0.9):
    model = model.to(device)
    model.train()

    train_loss = 0.0
    for X, y in dataloader:
        # 1. X, y 를 device로 이동 (model과 같은 device에 위치시킨다.)
        X, y = X.to(device), y.to(device)

        # 2. 추론
        pred = model(X, y, teacher_forcing_rate)

        # pred.shape: [batch_size, seq_length, vocab_size] 
        # shape변경  : [batch_size * seq_length, vocab_size] # 2차원으로 변경 ==> loss 계산
        y_hat = pred.reshape(-1, pred.shape[2])
        # 정답 shape: [batch_size, seq_length] -> [batch_size * seq_length] 2차원->1차원
        y = y.reshape(-1)
        # 3. loss 계산
        loss = loss_fn(y_hat, y) # [batch*seq_length, voca_size] , [batch*seq_length]
        # 4. grad 계산
        loss.backward()
        # 5. 파라미터 업데이트
        optimizer.step()
        # 6. grad 초기화
        optimizer.zero_grad()

        train_loss += loss.item()
    
    return train_loss / len(dataloader)

### Test 함수

In [87]:
@torch.no_grad
def eval(model, dataloader, loss_fn, device):
    model = model.to(device)
    model.eval()

    eval_loss = 0.0
    for X, y in dataloader:
        # X, y 를 device로 이동
        X, y = X.to(device), y.to(device)
        # 추론
        pred = model(X, y, teacher_forcing_rate=0.0)
        # loss계산
        y_hat = pred.reshape(-1, pred.shape[-1])
        y = y.reshape(-1)
        eval_loss += loss_fn(y_hat, y)

    return eval_loss / len(dataloader)

### Training

In [89]:
import os
os.makedirs("saved_model/chatbot_seq2seq", exist_ok=True)

In [ ]:
import time
epochs = 20
epochs = 1

model_save_path = "saved_model/chatbot_seq2seq/model.pth"
# 학습도중 loss가 계선되면 저장
best_loss = torch.inf

s = time.time()
for epoch in range(epochs):
    train_loss = train(model, train_loader, optimizer, loss_fn, device, teacher_forcing_rate)
    eval_loss = eval(model, test_loader, loss_fn)

    if best_loss > eval_loss: # 성능개선
        torch.save(model, model_save_path)
        print(f">>>>>>> {epoch + 1}에서 저장")
        best_loss = eval_loss

    print(f"{epoch + 1} 에폭: {train_loss}, {eval_loss}")

print("걸린시간:", time.time() - s, "초")

# 결과확인

- Sampler:
    -  DataLoader가 Datatset의 값들을 읽어서 batch를 만들때 index 순서를 정해주는 객체.
    -  DataLoader의 기본 sampler는 SequentialSampler 이다. shuffle=True 일경우 RandomSampler: 랜덤한 순서로 제공.

In [ ]:
def handle_special_tokens(encoded_string):
    # subword 토큰 처리
    # "나는 청 ##소년 이다." -> "나는 청소년 이다"
    tokens = encoded_string.split() # ["나는", "청", "##소년", "이다."]
    new_tokens = []
    for token in tokens:
        if token.startswith("##"):
            if new_tokens: 
                new_tokens[-1] += token[2:]
            else: # new_tokens의 len가 0 => ##XXX 첫번째 토큰
                new_tokens.append(token[2:]) # ##을 지우고 list에 추가.
        else: # 어절의 시작 토큰 -> " "+토큰
            if new_tokens:
                new_tokens.append(" "+token)
            else:
                new_tokens.append(token)
    return "".join(new_tokens)

In [ ]:
from torch.utils.data import SubsetRandomSampler
# sampler는 "Dataset에서 어떤 순서로 index를 뽑을지 결정하는 객체"이다. 
## DataLoader의 기본 Sampler는 shuffle=True 이면 RandomSampler(index를 섞어서제공)가 사용된다. False이면 SequentialSampler(index 순서대로 제공)가 사용된다.

def random_evaluation(model, dataset, device, n=10):
    """
    Dataset에서 일부 질문-답변 쌍들을 가져다 모델에 질문을 넣어 추론한 결과와 함께 확인.
    Parameter
        model: 학습된 seq2seq 모델
        dataset: 질문-답변 쌍울 추출할 dataset
        device
        n: int - 추출할 질문-답변 쌍 개수 default: 10
    """
    ## 평가할 데이터셋을 만들기
    n_samples = len(dataset)       # Dataset의 총 데이터개수
    # index = list(range(n_samples)) # Dataset의 index만들기.  [0, 1, 2, ...., dataset_length]
    # np.random.shuffle(index)       # 값들을 랜덤하게 섞어준다. [100, 23, 590, 10, ...]
    # sample_index = index[ : n]     # 평가할 데이터 개수만큼 index 생성.

    sample_index = torch.randint(0, n_samples, size=[n])

    # Dataloader 생성
    # SubsetRandomSampler: 지정한 index들 안에서 random한 순서로 제공.
    sampler = SubsetRandomSampler(sample_index)
    sample_loader = DataLoader(dataset, batch_size=n, sampler=sampler)

    ## 추론 후 확인
    model.to(device)
    model.eval()
    with torch.no_grad():
        for X, y in sample_loader:
            X, y = X.to(device), y.to(device)
            output = model(X, y, 0.0) # [batch, seq_len, vocab_size]

            # torch.Tensor -> ndarray : tokenizer decode에 넣기 위해.
            ## tensor를 cpu로 이동후 변환가능.
            ### tensor가 grad를 가지고 있으면(계산그래프에 포함되 있으면) -> tensor.detach().cpu().ndarray()

            pred = output.cpu().numpy()  # X.to("cpu") # 모델 추정한 답변
            X = X.cpu().numpy()   # 정답의 질문
            y = y.cpu().numpy()   # 정답의 답변 (batch, seq_len, vocab)

            for i in range(n):
                q = handle_special_tokens(tokenizer.decode(X[i]))
                a = handle_special_tokens(tokenizer.decode(y[i]))
                p = handle_special_tokens(tokenizer.decode(pred[i].argmax(-1)))
                print(f"질문: {q}")
                print(f"정답: {a}")
                print(f"예측: {p}")
                print('==================================================')

# 학습모델을 이용한 대화